# Bucket Batch Debug

This notebook inspects the stage-2 SRV training dataset as loaded by the local training pipeline.

It checks two things:

1. Whether stored bucket entries contain only one qubit count.
2. Whether the effective batch seen by `cut_padding_Bucket_collate_fn` still contains only one qubit count.

The goal is to verify that bucket padding really forms same-qubit batches, as intended by the paper.

In [ ]:
import os
import sys
from collections import Counter
from pathlib import Path

import hydra
import torch
from hydra.core.global_hydra import GlobalHydra

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from quantum_diffusion.data import DatasetLoader
from my_genQC.utils.misc_utils import infer_torch_device

In [ ]:
def compose_training_cfg(preset: str, extra_overrides: list[str] | None = None):
    overrides = [f"training={preset}"]
    if extra_overrides:
        overrides.extend(extra_overrides)

    GlobalHydra.instance().clear()
    with hydra.initialize_config_dir(version_base=None, config_dir=str(PROJECT_ROOT / "conf")):
        cfg = hydra.compose(config_name="config.yaml", overrides=overrides)

    return cfg["training"]


def infer_qubit_counts_from_x(x: torch.Tensor, pad_constant: int) -> torch.Tensor:
    non_pad = x != pad_constant
    counts = non_pad.any(dim=2).sum(dim=1).to(torch.int64)
    counts[counts == 0] = 1
    return counts


def infer_gate_lengths_from_x(x: torch.Tensor, pad_constant: int) -> torch.Tensor:
    non_pad = x != pad_constant
    lengths = non_pad.any(dim=1).sum(dim=1).to(torch.int64)
    lengths[lengths == 0] = 1
    return lengths


def summarize_bucket_dataset(dataset):
    if not hasattr(dataset, "z"):
        raise ValueError("Dataset has no z metadata; cannot inspect bucket structure.")

    z = dataset.z.cpu()
    if z.ndim != 3:
        raise ValueError(f"Expected bucketed z with rank 3, got shape {tuple(z.shape)}")

    rows = []
    for bucket_idx in range(z.shape[0]):
        qubit_counts = sorted({int(v) for v in z[bucket_idx, :, 0].tolist()})
        gate_lengths = sorted({int(v) for v in z[bucket_idx, :, 1].tolist()})
        rows.append(
            {
                "bucket_idx": bucket_idx,
                "unique_qubit_counts": qubit_counts,
                "num_unique_qubit_counts": len(qubit_counts),
                "min_gate_len": min(gate_lengths),
                "max_gate_len": max(gate_lengths),
                "num_unique_gate_lengths": len(gate_lengths),
            }
        )

    mixed = [row for row in rows if row["num_unique_qubit_counts"] > 1]
    pure_counter = Counter(
        row["unique_qubit_counts"][0] for row in rows if row["num_unique_qubit_counts"] == 1
    )

    return {
        "rows": rows,
        "mixed": mixed,
        "pure_counter": pure_counter,
        "num_buckets": len(rows),
        "num_mixed_buckets": len(mixed),
    }


def emulate_bucket_collate(dataset, bucket_idx: int):
    item = (dataset.x[bucket_idx], dataset.y[bucket_idx], dataset.z[bucket_idx])
    x_cut, y_cut = dataset.cut_padding_Bucket_collate_fn([item])
    pad_constant = int(dataset.params_config.pad_constant)

    stored_qubit_counts = sorted({int(v) for v in dataset.z[bucket_idx, :, 0].tolist()})
    stored_gate_lengths = sorted({int(v) for v in dataset.z[bucket_idx, :, 1].tolist()})
    inferred_qubit_counts = sorted({int(v) for v in infer_qubit_counts_from_x(x_cut, pad_constant).tolist()})
    inferred_gate_lengths = sorted({int(v) for v in infer_gate_lengths_from_x(x_cut, pad_constant).tolist()})

    return {
        "bucket_idx": bucket_idx,
        "x_cut_shape": tuple(x_cut.shape),
        "stored_qubit_counts": stored_qubit_counts,
        "inferred_qubit_counts": inferred_qubit_counts,
        "stored_gate_length_range": (min(stored_gate_lengths), max(stored_gate_lengths)),
        "inferred_gate_length_range": (min(inferred_gate_lengths), max(inferred_gate_lengths)),
    }


def print_bucket_report(report, max_examples: int = 10):
    print(f"num_buckets       : {report['num_buckets']}")
    print(f"num_mixed_buckets : {report['num_mixed_buckets']}")
    print(f"pure bucket counts: {dict(sorted(report['pure_counter'].items()))}")

    if report["mixed"]:
        print("\nMixed bucket examples:")
        for row in report["mixed"][:max_examples]:
            print(row)
    else:
        print("\nAll stored buckets are pure by qubit count.")

In [ ]:
TRAINING_PRESET = "paper_srv_bucket"
EXTRA_OVERRIDES = []
MAX_BUCKETS_TO_EMULATE = 32

In [ ]:
cfg = compose_training_cfg(TRAINING_PRESET, EXTRA_OVERRIDES)
device = infer_torch_device() if cfg.general.device == "auto" else cfg.general.device

print(f"preset                : {TRAINING_PRESET}")
print(f"dataset               : {cfg.general.dataset}")
print(f"padding_mode          : {cfg.training.get('padding_mode', 'max')}")
print(f"configured batch_size : {cfg.training.batch_size}")
print(f"device                : {device}")

dataset_loader = DatasetLoader(config=cfg, device=device)
dataset = dataset_loader.load_dataset(cfg.general.dataset, load_embedder=False)

print(f"dataset type          : {type(dataset).__name__}")
print(f"x shape               : {tuple(dataset.x.shape)}")
print(f"z shape               : {tuple(dataset.z.shape) if hasattr(dataset, 'z') else None}")
print(f"bucket_batch_size     : {getattr(dataset, 'bucket_batch_size', None)}")
print(f"collate_fn            : {getattr(dataset, 'collate_fn', None)}")
print(f"pad_constant          : {int(dataset.params_config.pad_constant)}")

In [ ]:
report = summarize_bucket_dataset(dataset)
print_bucket_report(report)

assert report["num_mixed_buckets"] == 0, (
    f"Found {report['num_mixed_buckets']} mixed stored buckets; bucket training is not pure by qubit count."
)

In [ ]:
emulated = [
    emulate_bucket_collate(dataset, bucket_idx)
    for bucket_idx in range(min(MAX_BUCKETS_TO_EMULATE, dataset.x.shape[0]))
]

mixed_after_collate = [
    row for row in emulated if len(row["inferred_qubit_counts"]) > 1
]

print(f"emulated buckets checked : {len(emulated)}")
print(f"mixed after collate      : {len(mixed_after_collate)}")

for row in mixed_after_collate[:10]:
    print(row)

assert not mixed_after_collate, "At least one emulated bucket batch still mixes qubit counts after collate."

In [ ]:
print("First few emulated bucket batches:")
for row in emulated[:10]:
    print(row)